# Bineral VSS
Author: Yifan Deng

## Check formatting and imports

In [113]:
import numpy as np
import soundfile as sf
from scipy.signal import fftconvolve
from scipy.io import loadmat
from IPython.display import Audio, display

In [114]:
# check keys
data = loadmat("IRC_1018/COMPENSATED/MAT/HRIR/IRC_1018_C_HRIR.mat")
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'r_eq_hrir_S', 'l_eq_hrir_S'])


In [115]:
l_hrirs = data['l_eq_hrir_S']
r_hrirs = data['r_eq_hrir_S']
print(l_hrirs.dtype)
print(l_hrirs.shape)

[('elev_v', 'O'), ('azim_v', 'O'), ('type_s', 'O'), ('sampling_hz', 'O'), ('content_m', 'O')]
(1, 1)


## Load surround files

In [116]:
audio_folder = "surround_sound"
channels = {
    "FrontLeft":  f"{audio_folder}/FrontLeft.wav",
    "FrontRight": f"{audio_folder}/FrontRight.wav",
    "Center":     f"{audio_folder}/Center.wav",
    "RearLeft":   f"{audio_folder}/RearLeft.wav",
    "RearRight":  f"{audio_folder}/RearRight.wav",
}

In [138]:
print("Play original by channels:")
print("FrontLeft")
display(Audio(f"{audio_folder}/FrontLeft.wav"))
print("FrontRight:")
display(Audio(f"{audio_folder}/FrontRight.wav"))
print("Center:")
display(Audio(f"{audio_folder}/Center.wav"))
print("RearLeft:")
display(Audio(f"{audio_folder}/RearLeft.wav"))
print("RearRight:")
display(Audio(f"{audio_folder}/RearRight.wav"))

Play original by channels:
FrontLeft


FrontRight:


Center:


RearLeft:


RearRight:


In [117]:
audio_data = {}
min_len = float('inf')

In [118]:
for ch, path in channels.items():
    sig, sr = sf.read(path)
    audio_data[ch] = sig
    min_len = min(min_len, len(sig))

In [119]:
# trim
for ch in audio_data:
    audio_data[ch] = audio_data[ch][:min_len]

In [120]:
# 3D mapping
channel_directions = {
    "FrontLeft":  (330, 0),
    "FrontRight": (30, 0),
    "Center":     (0, 0),
    "RearLeft":   (240, 0),
    "RearRight":  (120, 0),
}

## Load and apply HRTFs

In [121]:
mat_file = "IRC_1018/COMPENSATED/MAT/HRIR/IRC_1018_C_HRIR.mat"
data = loadmat(mat_file)

In [122]:
l_struct = data['l_eq_hrir_S'][0, 0]
r_struct = data['r_eq_hrir_S'][0, 0]

left_hrirs = l_struct['content_m']       # shape: (187, 512)
right_hrirs = r_struct['content_m']
azimuths = l_struct['azim_v'].flatten()
elevations = l_struct['elev_v'].flatten()
fs = int(l_struct['sampling_hz'].flatten()[0])

In [123]:
# get HRIR by direction
def find_hrir(azimuth, elevation):
    az = azimuth % 360
    el = elevation % 360
    dists = np.abs(azimuths - az) + np.abs(elevations - el)
    idx = np.argmin(dists)
    return left_hrirs[idx].flatten(), right_hrirs[idx].flatten()

In [124]:
def apply_hrtf(signal, hrtf_l, hrtf_r):
    left = fftconvolve(signal, hrtf_l, mode='full')[:len(signal)]
    right = fftconvolve(signal, hrtf_r, mode='full')[:len(signal)]
    return np.vstack((left, right))

## VSS stereo output

In [125]:
vss_mix = np.zeros((2, min_len))

for ch, sig in audio_data.items():
    az, el = channel_directions[ch]
    hrir_l, hrir_r = find_hrir(az, el)
    stereo = apply_hrtf(sig, hrir_l, hrir_r)
    vss_mix += stereo

In [126]:
# normalize and save
vss_mix /= np.max(np.abs(vss_mix))
sf.write("output_vss.wav", vss_mix.T, sr)
print("output_vss.wav created")

output_vss.wav created


## Downmix without HRTF

In [128]:
L = audio_data["FrontLeft"]
R = audio_data["FrontRight"]
C = audio_data["Center"]
RL = audio_data["RearLeft"]
RR = audio_data["RearRight"]

In [129]:
downmix_L = 0.3254 * L + 0.2301 * C + 0.2301 * RL
downmix_R = 0.3254 * R + 0.2301 * C + 0.2301 * RR

In [130]:
downmix = np.vstack((downmix_L, downmix_R))
downmix /= np.max(np.abs(downmix))
sf.write("output_downmix.wav", downmix.T, sr)
print("output_downmix.wav created")

output_downmix.wav created


In [131]:
print("Play VSS output:")
display(Audio("output_vss.wav"))
print("Playing stereo downmix:")
display(Audio("output_downmix.wav"))

Play VSS output:


Playing stereo downmix:
